# 🍽️ POC v8 — Photo de plat → 3D avec TRELLIS 2 (ton fork GitHub Omazi)

**Réglage :** `Exécution > Modifier le type d'exécution > A100 GPU` + **RAM élevée**.

- **1re fois** : il faut le secret Colab `HF_TOKEN` (🔑 à gauche) pour télécharger les modèles, puis la cellule *Sauvegarde* les copie sur ton Drive.
- **Ensuite** : les modèles sont chargés depuis ton Drive, plus besoin de Hugging Face.
- Le code vient de **github.com/Omazi** (TRELLIS.2, CuMesh, FlexGEMM) : les correctifs y sont déjà.

In [ ]:
# CELLULE 1 — GPU + Drive + source des modèles
import os, torch
from google.colab import drive
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'AUCUN'
mem = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
print(f'GPU : {gpu} ({mem:.0f} Go) | torch {torch.__version__}')
assert mem >= 22, "❌ Il faut un A100 : Exécution > Modifier le type d'exécution."

drive.mount('/content/drive')
BACKUP = '/content/drive/MyDrive/trellis2_backup'
HF_CACHE = f'{BACKUP}/hf_cache'
if os.path.exists(f'{HF_CACHE}/models--microsoft--TRELLIS.2-4B'):
    os.environ['HF_HUB_CACHE'] = HF_CACHE
    os.environ['HF_HUB_OFFLINE'] = '1'
    print('✅ Modèles chargés depuis ton Drive (hors ligne, sans Hugging Face)')
else:
    from google.colab import userdata
    from huggingface_hub import login
    try:
        login(userdata.get('HF_TOKEN')); print('✅ Jeton Hugging Face OK — pense à lancer la cellule Sauvegarde après la génération')
    except Exception:
        raise SystemExit('❌ Pas de sauvegarde Drive et pas de secret HF_TOKEN.')

In [ ]:
# CELLULE 2 — Installation depuis ton GitHub (15-25 min la 1re fois, ~1 min ensuite)
import os, sys, subprocess, glob, re, torch

def sh(cmd):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.returncode:
        errs = [l for l in (r.stdout + r.stderr).splitlines() if re.search(r'error|fatal|killed', l, re.I) and 'Werror' not in l]
        print('\n'.join(dict.fromkeys(errs[:60])) or (r.stdout + r.stderr)[-3000:])
        raise SystemExit(f'❌ Échec : {cmd[:120]} — copie-moi les lignes ci-dessus.')
    return r.stdout

sh('pip uninstall -y -q tensorflow tf-keras tensorflow-hub || true')
if not os.path.exists('/content/TRELLIS.2'):
    sh('git clone -q --recursive https://github.com/Omazi/TRELLIS.2.git /content/TRELLIS.2')
else:
    sh('git -C /content/TRELLIS.2 pull -q')
sh('pip install -q easydict plyfile zstandard kornia timm imageio imageio-ffmpeg trimesh rembg onnxruntime ninja ultralytics')
sh('pip install -q git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec8')
# Node.js >= 18 pour l'optimisation mobile du GLB
nv = subprocess.run('node -v', shell=True, text=True, capture_output=True).stdout.strip()
if not nv or int(nv.lstrip('v').split('.')[0]) < 18:
    sh('curl -fsSL https://nodejs.org/dist/v22.11.0/node-v22.11.0-linux-x64.tar.xz | tar -xJ -C /usr/local --strip-components=1')
sh('cd /content/TRELLIS.2/tools/mobile_glb && npm install -q --no-audit --no-fund')
print('✅ Node', subprocess.run('node -v', shell=True, text=True, capture_output=True).stdout.strip())

# Toolkit CUDA de même version majeure que PyTorch
tcuda = torch.version.cuda
def nvcc_ver(home):
    out = subprocess.run(f'{home}/bin/nvcc --version', shell=True, text=True, capture_output=True).stdout
    m = re.search(r'release (\d+\.\d+)', out); return m.group(1) if m else ''
match = [h for h in sorted(glob.glob('/usr/local/cuda-*')) + ['/usr/local/cuda'] if nvcc_ver(h).split('.')[0] == tcuda.split('.')[0]]
if not match:
    maj, mi = tcuda.split('.')
    subprocess.run(f'apt-get -qq update && apt-get -qq install -y cuda-toolkit-{maj}-{mi}', shell=True)
    match = [h for h in sorted(glob.glob('/usr/local/cuda-*')) if nvcc_ver(h).split('.')[0] == maj]
    if not match: raise SystemExit(f'❌ Toolkit CUDA {tcuda} introuvable.')
os.environ['CUDA_HOME'] = match[-1]
os.environ['PATH'] = f'{match[-1]}/bin:' + os.environ['PATH']
cap = torch.cuda.get_device_capability()
os.environ['TORCH_CUDA_ARCH_LIST'] = f'{cap[0]}.{cap[1]}'
os.environ['MAX_JOBS'] = str(os.cpu_count())

# Composants GPU : wheels sauvegardés sur Drive, sinon compilation
tag = f'torch{torch.__version__.replace("+","_")}_py{sys.version_info[0]}{sys.version_info[1]}_sm{cap[0]}{cap[1]}'
cache = f'/content/drive/MyDrive/trellis2_wheels/{tag}' if os.path.ismount('/content/drive') else f'/content/wheels/{tag}'
os.makedirs(cache, exist_ok=True)
exts = {
    'nvdiffrast': 'git clone -q -b v0.4.0 https://github.com/NVlabs/nvdiffrast.git /tmp/ext/nvdiffrast',
    'cumesh':     'git clone -q --recursive https://github.com/Omazi/CuMesh.git /tmp/ext/cumesh',
    'flex_gemm':  'git clone -q --recursive https://github.com/Omazi/FlexGEMM.git /tmp/ext/flex_gemm',
    'o_voxel':    'cp -r /content/TRELLIS.2/o-voxel /tmp/ext/o_voxel',
}
os.makedirs('/tmp/ext', exist_ok=True)
for name, get in exts.items():
    found = glob.glob(f'{cache}/{name.replace("_","*")}*.whl') + glob.glob(f'{cache}/{name}*.whl')
    if found:
        sh(f'pip install -q --no-deps --force-reinstall "{found[0]}"'); print(f'✅ {name} (déjà compilé)'); continue
    print(f'🔨 Compilation de {name}...')
    if not os.path.exists(f'/tmp/ext/{name}'): sh(get)
    sh(f'pip wheel -v --no-build-isolation --no-deps /tmp/ext/{name} -w {cache}')
    whl = sorted(glob.glob(f'{cache}/*.whl'), key=os.path.getmtime)[-1]
    sh(f'pip install -q --no-deps "{whl}"'); print(f'✅ {name} compilé')
print('🎉 Installation terminée')

In [ ]:
# CELLULE 3 — Photo + détourage
# MODE = 'tout' : garde le contenant, le papier, les sauces (comme Tripo) ; tu cliques sur ce qu'il faut RETIRER.
# MODE = 'plat' : garde uniquement la nourriture (automatique).
MODE = 'tout'

import sys, io, base64
sys.path.insert(0, '/content/TRELLIS.2')
from google.colab import files, output
from IPython.display import HTML, JSON, display
from PIL import Image
from trellis2.utils.dish_segmentation import ClickSegmenter, cut_dish

up = files.upload()
name = list(up.keys())[0]
img = Image.open(io.BytesIO(up[name])).convert('RGB'); img.thumbnail((1536, 1536))
OUT = '/content/input_rgba.png'

if MODE == 'plat':
    try:
        cut = cut_dish(img)
    except Exception:
        import rembg; cut = cut_dish(img, rembg.new_session('isnet-general-use'))
    cut.save(OUT)
    prev = Image.new('RGB', cut.size, (70, 170, 70)); prev.paste(cut, mask=cut.split()[3]); prev.thumbnail((500, 500)); display(prev)
    print('✅ Détourage automatique fait')
else:
    seg = ClickSegmenter(img)
    seg.rgba().save(OUT)

    def _png():
        b = io.BytesIO(); seg.preview(700).save(b, 'PNG'); return base64.b64encode(b.getvalue()).decode()

    def _reply():
        seg.rgba().save(OUT); return JSON({'img': _png(), 'kept': round(100 * float(seg.mask.mean()))})

    def _click(x, y, remove):
        seg.click(int(x), int(y), bool(remove)); return _reply()

    def _undo():
        seg.undo(); return _reply()

    output.register_callback('dish_click', _click)
    output.register_callback('dish_undo', _undo)
    W, H = seg.image.size
    html = '''
<div style="font-family:system-ui,sans-serif;max-width:700px">
  <div style="margin:6px 0 10px;display:flex;gap:8px;align-items:center;flex-wrap:wrap">
    <button id="dmode" style="padding:7px 12px;border-radius:8px;border:0;background:#c4501f;color:#fff;font-weight:600;cursor:pointer">Mode : RETIRER</button>
    <button id="dundo" style="padding:7px 12px;border-radius:8px;border:1px solid #bbb;background:#fff;cursor:pointer">Annuler le dernier clic</button>
    <span id="dstat" style="color:#555;font-size:14px">Clique sur la table / le fond à retirer. Le vert = retiré.</span>
  </div>
  <img id="dimg" src="data:image/png;base64,__IMG__" style="max-width:100%;cursor:crosshair;border-radius:8px;display:block">
</div>
<script>
(function () {
  const img = document.getElementById('dimg'), stat = document.getElementById('dstat'), modeBtn = document.getElementById('dmode');
  let remove = true, busy = false;
  modeBtn.onclick = () => {
    remove = !remove;
    modeBtn.textContent = 'Mode : ' + (remove ? 'RETIRER' : 'RAJOUTER');
    modeBtn.style.background = remove ? '#c4501f' : '#2e7d32';
  };
  async function call(fn, args) {
    if (busy) return;
    busy = true; stat.textContent = 'Découpage en cours…';
    try {
      const res = await google.colab.kernel.invokeFunction(fn, args, {});
      const d = res.data['application/json'];
      img.src = 'data:image/png;base64,' + d.img;
      stat.textContent = 'Gardé : ' + d.kept + ' % de la photo. Continue, ou passe à la cellule 4.';
    } catch (e) { stat.textContent = 'Erreur : ' + e; }
    busy = false;
  }
  img.onclick = (e) => {
    const r = img.getBoundingClientRect();
    const x = Math.round((e.clientX - r.left) / r.width * __W__);
    const y = Math.round((e.clientY - r.top) / r.height * __H__);
    call('dish_click', [x, y, remove]);
  };
  document.getElementById('dundo').onclick = () => call('dish_undo', []);
})();
</script>'''
    display(HTML(html.replace('__IMG__', _png()).replace('__W__', str(W)).replace('__H__', str(H))))
    print('👆 Clique sur la photo ci-dessus. Le résultat est enregistré à chaque clic.')

In [ ]:
# CELLULE 4 — Génération 3D haute définition (~1-2 min, + chargement des modèles la 1re fois)
import os, sys, time, torch
sys.path.insert(0, '/content/TRELLIS.2'); os.chdir('/content/TRELLIS.2')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '1'
from PIL import Image
from trellis2.pipelines import Trellis2ImageTo3DPipeline
if 'pipeline' not in globals():
    pipeline = Trellis2ImageTo3DPipeline.from_pretrained('microsoft/TRELLIS.2-4B', load_rembg=False)
    pipeline.cuda()
t = time.time()
mesh = pipeline.run(Image.open('/content/input_rgba.png'), seed=42, pipeline_type='1536_cascade')[0]
print(f'✅ Généré en {time.time()-t:.0f} s')

In [ ]:
# CELLULE 5 — Export GLB : fond fermé + optimisation mobile + téléchargement
import o_voxel, os, subprocess, sys
from google.colab import files
sys.path.insert(0, '/content/TRELLIS.2')
from trellis2.utils.food_postprocess import solidify_base
TAILLE_M = 0.30   # plus grand côté du plat en mètres (taille réelle en AR)

mesh.simplify(16777216)
glb = o_voxel.postprocess.to_glb(
    vertices=mesh.vertices, faces=mesh.faces, attr_volume=mesh.attrs, coords=mesh.coords,
    attr_layout=mesh.layout, voxel_size=mesh.voxel_size, aabb=[[-0.5, -0.5, -0.5], [0.5, 0.5, 0.5]],
    decimation_target=300000, texture_size=4096, remesh=True, remesh_band=1, remesh_project=0, verbose=True,
)
glb.vertices, _ = solidify_base(glb.vertices, glb.faces)        # dessous rabattu sur la table
glb.export('/content/plat_hd.glb')                               # version pleine qualité (archive)
r = subprocess.run(f'cd /content/TRELLIS.2/tools/mobile_glb && node optimize.mjs /content/plat_hd.glb /content/plat_mobile.glb {TAILLE_M} 0.35',
                   shell=True, text=True, capture_output=True)
print(r.stdout[-500:], r.stderr[-1500:] if r.returncode else '')
for f in ['/content/plat_hd.glb', '/content/plat_mobile.glb']:
    print(f'✅ {os.path.basename(f)} : {os.path.getsize(f)/1e6:.1f} Mo')
files.download('/content/plat_mobile.glb')

In [ ]:
# CELLULE 6 — Sauvegarde des modèles sur Drive (une seule fois, ~18 Go, 5-10 min)
import os, subprocess
if os.environ.get('HF_HUB_OFFLINE') == '1':
    print('✅ Déjà chargés depuis ta sauvegarde Drive, rien à faire')
else:
    BACKUP = '/content/drive/MyDrive/trellis2_backup'; os.makedirs(BACKUP, exist_ok=True)
    src = os.path.expanduser('~/.cache/huggingface/hub')
    subprocess.run(f'rsync -a "{src}/" "{BACKUP}/hf_cache/"', shell=True, check=True)
    print(subprocess.run(f'du -sh {BACKUP}/hf_cache', shell=True, capture_output=True, text=True).stdout)
    print('✅ Modèles sauvegardés : les prochaines sessions n’auront plus besoin de Hugging Face')